# AutoData — Strict External Fraud Evaluation (fraudTrain → fraudTest)

This notebook is the **external holdout checkpoint** after the 100k leakage-audited validation.

Rules:
1. `fraudTrain.csv` is the **only** development source.
2. `fraudTest.csv` is assigned **only** to the external test split.
3. Model selection and threshold tuning use a temporal validation slice from `fraudTrain.csv`.
4. Tokenizer, numeric scaling, missing-value handling, rare-category grouping, and quality-dependent decisions are fit on development **train rows only**.
5. E2 behavioral features may use unsampled transactions that occurred **strictly earlier** than the row being scored — this mirrors a live transaction database — but never future rows or labels.
6. No threshold or feature choice is tuned on `fraudTest.csv`.

Primary metric: **external PR-AUC**. Precision/recall/F1 use the threshold selected on development validation only.

In [ ]:
# 0) GPU check
import os, sys, json, copy, zipfile
from pathlib import Path
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi || true
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > T4/L4/A100.")

In [ ]:
# 1) Locate/upload the latest project ZIP
CONTENT = Path("/content")
PROJECT_ROOT = CONTENT / "transaction-data-intelligence"
if not (PROJECT_ROOT / "src").exists():
    candidates = list(CONTENT.glob("*.zip"))
    zpath = candidates[0] if len(candidates) == 1 else None
    if zpath is None:
        from google.colab import files
        print("Upload AutoData_v2_external_eval.zip (or the latest project ZIP)")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No project ZIP uploaded")
        zpath = CONTENT / next(iter(uploaded))
    print("Extracting", zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(CONTENT)
if not (PROJECT_ROOT / "src").exists():
    matches = [p.parent for p in CONTENT.rglob("config.yaml") if (p.parent / "src").exists()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not uniquely locate project root: {matches}")
    PROJECT_ROOT = matches[0]
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# 2) Install repository dependencies
!pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available()
print("GPU ready:", torch.cuda.get_device_name(0))

## Data

Upload **both original files**. Do not substitute a preprocessed Kaggle copy. The notebook checks that the external source starts after the development source ends.

In [ ]:
# 3) Controls — T4-safe defaults
TRAIN_PATH = "/content/fraudTrain.csv"
EXTERNAL_PATH = "/content/fraudTest.csv"
TARGET = "is_fraud"

TRAIN_SAMPLE_ROWS = 70_000
VALIDATION_SAMPLE_ROWS = 15_000
EXTERNAL_TEST_ROWS = 100_000   # set to "full" after this checkpoint if Colab RAM permits
TRAIN_POSITIVE_SHARE = 0.10
DEV_TRAIN_FRACTION = 0.85

SEEDS = [42, 123, 456]
EPOCHS = 10
PATIENCE = 4
BATCH_SIZE = 256

print({
    "train_sample": TRAIN_SAMPLE_ROWS, "validation_sample": VALIDATION_SAMPLE_ROWS,
    "external_test": EXTERNAL_TEST_ROWS, "seeds": SEEDS, "epochs": EPOCHS, "batch": BATCH_SIZE
})

In [ ]:
# 4) Load both sources through AutoData; infer schema/roles from DEVELOPMENT ONLY
import pandas as pd, numpy as np
from src.ingestion.loader import load_dataset
from src.ingestion.roles import detect_roles, schema_for_profiling
from src.ingestion.schema_detector import detect_schema
from src.preprocessing.levels import DataPreparer, infer_roles
from src.profiling.profiler import parse_datetime_column
from src.utils.config import ROOT, load_config

cfg = load_config()

def ensure_file(path, label):
    p=Path(path)
    if p.exists(): return p
    from google.colab import files
    print(f"Upload {label}")
    uploaded=files.upload()
    if not uploaded: raise RuntimeError(f"Missing {label}")
    return Path('/content') / next(iter(uploaded))

train_path=ensure_file(TRAIN_PATH, 'fraudTrain.csv')
ext_path=ensure_file(EXTERNAL_PATH, 'fraudTest.csv')
train_ds=load_dataset(train_path, metadata_dir=ROOT / cfg['project']['data_dir'] / 'raw' / '_metadata')
ext_ds=load_dataset(ext_path, metadata_dir=ROOT / cfg['project']['data_dir'] / 'raw' / '_metadata')
train_df=train_ds.df.copy(); external_df=ext_ds.df.copy()

# Create stable source-scoped row IDs explicitly. load_dataset() does not guarantee
# that an internal _row_id column is present on every ingestion path.
# These IDs are metadata only; they are excluded from model features downstream.
train_df['_row_id'] = 'dev::' + pd.Series(np.arange(len(train_df)), index=train_df.index).astype(str)
external_df['_row_id'] = 'ext::' + pd.Series(np.arange(len(external_df)), index=external_df.index).astype(str)
assert train_df['_row_id'].is_unique and external_df['_row_id'].is_unique
assert set(train_df['_row_id']).isdisjoint(set(external_df['_row_id']))

missing_ext=set(train_df.columns)-set(external_df.columns)
missing_dev=set(external_df.columns)-set(train_df.columns)
if missing_ext or missing_dev:
    raise RuntimeError(f"Schema mismatch. Missing in external={missing_ext}; missing in development={missing_dev}")
external_df=external_df[train_df.columns]

# IMPORTANT: schema and role inference only see fraudTrain.csv.
schema=detect_schema(train_df, train_ds.metadata.dataset_id)
roles=detect_roles(train_df, schema, target=TARGET)
ps=schema_for_profiling(schema, roles, train_df)
level_roles=infer_roles(train_df, ps, roles)

print(f"development={train_path.name}: {len(train_df):,} rows")
print(f"external={ext_path.name}: {len(external_df):,} rows")
print(f"target={level_roles.target}, entity={level_roles.entity}, time={level_roles.time}, amount={level_roles.amount}")
assert level_roles.target == TARGET

In [ ]:
# 5) Verify chronological external holdout, combine only for point-in-time feature lookup
dev_time=parse_datetime_column(train_df[level_roles.time], 'datetime')
ext_time=parse_datetime_column(external_df[level_roles.time], 'datetime')
print('Development range:', dev_time.min(), '->', dev_time.max())
print('External range:   ', ext_time.min(), '->', ext_time.max())
if not (dev_time.max() < ext_time.min()):
    raise RuntimeError('External source is not strictly later than development source; do not use this as a chronological holdout.')

combined=pd.concat([train_df, external_df], ignore_index=True)
print('Combined history stream:', len(combined))
print('NOTE: E2 may consult unsampled earlier transactions in this stream, never later ones or labels.')

In [ ]:
# 6) Freeze a strict split: fraudTrain -> train/validation, fraudTest -> TEST ONLY
external_cfg=copy.deepcopy(cfg)
external_cfg['processing']['quality_fit_scope']='train'
external_cfg['processing']['strict_external_cleaning']=True
external_cfg['sampling']['train_positive_share']=TRAIN_POSITIVE_SHARE
external_cfg['sampling']['validation_positive_share']=None
external_cfg['sampling']['keep_all_test_positives']=False

cutoff=dev_time.quantile(DEV_TRAIN_FRACTION)
split_map=pd.Series(index=combined['_row_id'], dtype='object')
split_map.loc[train_df.loc[dev_time < cutoff, '_row_id']]='train'
split_map.loc[train_df.loc[dev_time >= cutoff, '_row_id']]='validation'
split_map.loc[external_df['_row_id']]='test'

external_n = len(external_df) if EXTERNAL_TEST_ROWS == 'full' else int(EXTERNAL_TEST_ROWS)
sizes={'train': int(TRAIN_SAMPLE_ROWS), 'validation': int(VALIDATION_SAMPLE_ROWS), 'test': external_n}
bounds={
    'method':'fixed external source holdout',
    'development_train':f"{dev_time.min()} to before {cutoff}",
    'development_validation':f"{cutoff} to {dev_time.max()}",
    'external_test':f"{ext_time.min()} to {ext_time.max()}",
    'external_source':ext_path.name,
}

def make_preparer(config):
    p=DataPreparer(combined, ps, level_roles, config, rows='full', seed=cfg['project']['seed'])
    info=p.prepare_fixed_split(split_map, sizes, positive_share={'train':TRAIN_POSITIVE_SHARE}, boundaries=bounds)
    return p, info

probe, split_info=make_preparer(external_cfg)
print(json.dumps(split_info['sample'], indent=2, default=str))
assert set(probe.sample_ids.loc[probe.sample_ids['_split']=='test','_row_id'].str[:5]) == {'ext::'}
assert not probe.sample_ids.loc[probe.sample_ids['_split'].isin(['train','validation']),'_row_id'].str.startswith('ext::').any()
print('PASS: external rows are test-only.')

In [ ]:
# 7) Build E2 once and require point-in-time audit
e2_probe=probe.build('E2')
pit=e2_probe.info.get('point_in_time', {})
print(json.dumps(pit, indent=2, default=str))
assert pit.get('passed'), pit
for k in ['card_history','merchant_history','previous_transactions']:
    if k in pit:
        assert pit[k]['passed']
        assert pit[k].get('same_timestamp_policy') == 'strictly_earlier_only'
print('PASS: history/sequence features are strictly point-in-time.')

## Frozen comparison

These variants were selected **before seeing external results**, from the 100k development validation. `fraudTest.csv` is not used to add/remove variants.

In [ ]:
# 8) Train on development, select threshold on development validation, score external test
import time
from src.models.sanity_transformer import SanityTransformerAdapter
from src.evaluation.metrics import classification_metrics

settings={'epochs':EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE}
variant_specs=[
    ('E0_raw','E0',None),
    ('E1_continuous','E1','continuous'),
    ('E2_full','E2',['temporal','history','sequence']),
    ('E2_sequence','E2',['sequence']),
    ('E2_history_sequence','E2',['history','sequence']),
]

all_rows=[]
for variant, level, mode in variant_specs:
    vcfg=copy.deepcopy(external_cfg)
    if variant == 'E1_continuous':
        vcfg['representation']['numeric_mode']='continuous'
        vcfg['representation']['numeric_coarse_bins']=0
    if level == 'E2':
        vcfg['features']['enabled_groups']=mode
    p,_=make_preparer(vcfg)
    prepared=p.build(level)
    if level == 'E2':
        assert prepared.info.get('point_in_time',{}).get('passed')
    print(f"\n=== {variant}: features={len(prepared.features)} ===")
    for seed in SEEDS:
        adapter=SanityTransformerAdapter(vcfg, device='cuda')
        summary=adapter.train(prepared, {**settings,'seed':seed})
        results,pv,pt=adapter.evaluate(prepared)
        m=results['test_unweighted']
        vm=results['validation']
        row={
            'variant':variant,'level':level,'seed':seed,'features':len(prepared.features),
            'threshold_from_validation':results['threshold_from_validation'],
            'epochs_run':summary['epochs_run'],'best_val_pr_auc':summary['best_val_pr_auc'],
            'external_n':m['n'],'external_positives':m['positives'],
            'external_pr_auc':m['pr_auc'],'external_roc_auc':m['roc_auc'],
            'external_precision':m['precision'],'external_recall':m['recall'],'external_f1':m['f1'],
            'external_log_loss':m['log_loss'],
            'validation_pr_auc_weighted':vm['pr_auc'],
        }
        all_rows.append(row)
        print(f" seed={seed} | external PR-AUC={m['pr_auc']:.6f} | P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} | threshold={row['threshold_from_validation']:.4f}")

results_df=pd.DataFrame(all_rows)
display(results_df)

In [ ]:
# 9) Aggregate stability — external holdout is never used for selection
metrics=['external_pr_auc','external_roc_auc','external_precision','external_recall','external_f1','external_log_loss']
agg=results_df.groupby('variant').agg(
    seeds=('seed','nunique'),
    features=('features','first'),
    **{f'{m}_mean':(m,'mean') for m in metrics},
    **{f'{m}_std':(m,'std') for m in metrics},
).reset_index().sort_values('external_pr_auc_mean', ascending=False)
display(agg)

print('Interpretation rule: external PR-AUC is the primary ranking metric. Threshold metrics use validation-selected thresholds only.')
print('Do NOT rerun feature selection based on fraudTest.csv; if we change the pipeline after seeing this table, a new later holdout is required.')

In [ ]:
# 10) Export reproducible results and context
from datetime import datetime
out_dir=PROJECT_ROOT/'experiments'/'external_gpu'
out_dir.mkdir(parents=True, exist_ok=True)
stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
long_path=out_dir/f'external_holdout_{stamp}.csv'
agg_path=out_dir/f'external_holdout_summary_{stamp}.csv'
ctx_path=out_dir/f'external_holdout_context_{stamp}.json'
results_df.to_csv(long_path,index=False); agg.to_csv(agg_path,index=False)
context={
    'development_dataset':train_ds.metadata.dataset_id, 'development_file':train_path.name,
    'external_dataset':ext_ds.metadata.dataset_id, 'external_file':ext_path.name,
    'strict_external_evaluation':True, 'schema_and_roles_fit_on':'development source only',
    'quality_fit_scope':'train', 'strict_external_cleaning':True,
    'history_policy':'strictly earlier only; may use unsampled prior transactions; never labels',
    'split':split_info, 'point_in_time_audit':pit,
    'seeds':SEEDS,'epochs':EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE,
    'device':torch.cuda.get_device_name(0),'variants':[x[0] for x in variant_specs],
    'external_test_rows':external_n, 'threshold_source':'development validation only',
}
ctx_path.write_text(json.dumps(context,indent=2,default=str))
print(long_path); print(agg_path); print(ctx_path)
from google.colab import files
files.download(str(long_path)); files.download(str(agg_path)); files.download(str(ctx_path))

## What to send back

Upload all three exported files. If E2 still beats E0/E1 on this untouched source, we can call the improvement externally validated **for this public dataset pair**. We still should not treat `fraudTest.csv` as reusable tuning data afterward.